# 01 — Filtrado de archivos crudos ENOE

Lee los archivos de la ENOE y extrae solo las columnas necesarias para el análisis.
Los resultados se guardan en `datos/filtrado/`.

## Sobre los datos crudos

Los archivos crudos **no están en el repositorio** por su tamaño (~150 MB por trimestre).
Están disponibles en:
- **Drive compartido (2005–2020 T1):** https://drive.google.com/drive/folders/11be4oY_4OuJC4nAsvXUafzn6PGuvP8L6
- **INEGI (2020 T2 en adelante):** https://www.inegi.org.mx/programas/enoe/15ymas/#microdatos

## Cómo correr este notebook
- **Colab:** se conecta a Drive automáticamente.
- **Jupyter local:** coloca los crudos en `datos/crudo/` o descárgalos con `gdown` (ver celda de descarga).

In [1]:
import pandas as pd
import os
import re
import gc
import glob
from pathlib import Path

In [2]:
import sys

# Detecta el entorno para usar las rutas correctas en cada caso
EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Corriendo en Colab')
else:
    print('Corriendo en Jupyter local')

Corriendo en Jupyter local


## Rutas

Ajusta si tus carpetas tienen nombres distintos.

In [3]:
# Rutas relativas a la raíz del repo (este notebook está en scripts/)
RAIZ = Path('..').resolve()
CARPETAS_ENTRADA = [str(RAIZ / 'datos' / 'crudo')]
CARPETA_SALIDA   = str(RAIZ / 'datos' / 'filtrado')

os.makedirs(CARPETA_SALIDA, exist_ok=True)

print('Carpetas de entrada:')
for c in CARPETAS_ENTRADA:
    print(f'  {"OK" if os.path.exists(c) else "NO ENCONTRADA"}: {c}')
print(f'Salida: {CARPETA_SALIDA}')

Carpetas de entrada:
  OK: C:\Users\andre\hackODS\LoboEnsambladores\datos\crudo
Salida: C:\Users\andre\hackODS\LoboEnsambladores\datos\filtrado


## Descarga de crudos desde Drive (solo Jupyter local)

Descomenta y ejecuta **una sola vez** si no tienes los archivos descargados localmente.

In [4]:
# -- DESCOMENTA SOLO SI NECESITAS DESCARGAR LOS CRUDOS --
#
# !pip install gdown -q
# import gdown
#
# os.makedirs('../datos/crudo', exist_ok=True)
# gdown.download_folder(
#     id='11be4oY_4OuJC4nAsvXUafzn6PGuvP8L6',  # ID de la carpeta compartida
#     output='../datos/crudo',
#     quiet=False,
#     use_cookies=False,
# )

## Catálogos

Los crudos guardan todo como números (1, 2, 3...). Estos diccionarios los convierten a texto.

In [5]:
ENTIDADES = {
    1:'Aguascalientes', 2:'Baja California', 3:'Baja California Sur',
    4:'Campeche', 5:'Coahuila', 6:'Colima', 7:'Chiapas', 8:'Chihuahua',
    9:'Ciudad de México', 10:'Durango', 11:'Guanajuato', 12:'Guerrero',
    13:'Hidalgo', 14:'Jalisco', 15:'México', 16:'Michoacán', 17:'Morelos',
    18:'Nayarit', 19:'Nuevo León', 20:'Oaxaca', 21:'Puebla', 22:'Querétaro',
    23:'Quintana Roo', 24:'San Luis Potosí', 25:'Sinaloa', 26:'Sonora',
    27:'Tabasco', 28:'Tamaulipas', 29:'Tlaxcala', 30:'Veracruz',
    31:'Yucatán', 32:'Zacatecas',
}
SEXO     = {1:'Hombre', 2:'Mujer'}
FORMAL   = {1:'Formal', 2:'Informal'}
NIV_INS  = {1:'Sin instrucción', 2:'Básica', 3:'Media superior', 4:'Superior', 5:'No especificado'}
SECTORES = {
    1:'Agropecuario', 2:'Industria extractiva', 3:'Manufactura',
    4:'Construcción', 5:'Comercio', 6:'Restaurantes y hospedaje',
    7:'Transportes y comunicaciones', 8:'Servicios financieros y profesionales',
    9:'Servicios sociales', 10:'Servicios diversos', 11:'Gobierno',
}
OCUPACION = {
    1:'Funcionarios y directivos', 2:'Profesionistas y técnicos',
    3:'Trabajadores de la educación', 4:'Trabajadores del arte',
    5:'Actividades administrativas', 6:'Comerciantes y vendedores',
    7:'Servicios personales', 8:'Actividades agrícolas',
    9:'Industria extractiva y construcción', 10:'Operadores de maquinaria',
    11:'Otras ocupaciones',
}

## Funciones auxiliares

In [6]:
def parsear_nombre(ruta):
    # Busca el patrón SDEMT{trim}{YY} al final del nombre
    # Funciona con SDEMT419.csv, ENOE_SDEMT120.csv y ENOEN_SDEMT422.csv
    nombre = os.path.splitext(os.path.basename(ruta))[0].upper()
    m = re.search(r'SDEMT(\d)(\d{2})$', nombre)
    if not m:
        return None
    trim, yy = int(m.group(1)), m.group(2)
    anio = int('20' + yy)
    if trim not in [1, 2, 3, 4] or not (2000 <= anio <= 2030):
        return None
    return trim, anio


def nombre_filtrado(ruta):
    info = parsear_nombre(ruta)
    if info is None:
        return None
    trim, anio = info
    return f'ENOE_T{trim}{str(anio)[-2:]}.csv'


def a_num(serie):
    # Los archivos de 2023+ guardan algunos campos numéricos como texto con comillas
    return pd.to_numeric(serie, errors='coerce')


def filtrar(ruta):
    encabezado = pd.read_csv(ruta, encoding='latin1', nrows=0)
    cols = list(encabezado.columns)

    # Estas columnas cambiaron de nombre en la versión 2023 de la ENOE
    col_est = 'cve_ent' if 'cve_ent' in cols else 'ent'
    col_fac = 'fac_tri' if 'fac_tri' in cols else 'fac'

    quiero = [
        col_est, col_fac, 'sex', 'eda',
        'c_ocu11c', 'rama_est2', 'niv_ins', 'anios_esc',
        'hrsocup', 'ingocup', 'ing_x_hrs', 'emp_ppal', 'clase2',
    ]
    cols_leer = [c for c in quiero if c in cols]
    df = pd.read_csv(ruta, encoding='latin1', low_memory=False, usecols=cols_leer)

    # Quedarse solo con población ocupada (clase2 == 1)
    if 'clase2' in df.columns:
        df = df[a_num(df['clase2']) == 1].copy()
        df.drop(columns=['clase2'], inplace=True)

    for c in [col_est, 'sex', 'c_ocu11c', 'rama_est2', 'niv_ins', 'emp_ppal', 'hrsocup', 'ingocup']:
        if c in df.columns:
            df[c] = a_num(df[c])

    # Unificar nombres para que todos los años sean consistentes
    df.rename(columns={col_est: 'cve_ent', col_fac: 'fac_tri'}, inplace=True, errors='ignore')

    # Ingreso por hora: usar el campo del INEGI si existe, calcularlo si no
    if 'ing_x_hrs' in df.columns:
        df['ingreso_hora'] = a_num(df['ing_x_hrs']).where(a_num(df['ing_x_hrs']) > 0)
    if 'ingocup' in df.columns and 'hrsocup' in df.columns:
        ing, hrs = a_num(df['ingocup']), a_num(df['hrsocup'])
        calc = (ing / (hrs * 4.33)).round(2)
        if 'ingreso_hora' not in df.columns:
            df['ingreso_hora'] = calc.where((ing > 0) & (hrs > 0))
        else:
            df['ingreso_hora'] = df['ingreso_hora'].fillna(calc.where((ing > 0) & (hrs > 0)))

    if 'ingocup' in df.columns:
        ing = a_num(df['ingocup'])
        df['salario_semanal'] = (ing / 4.33).round(2).where(ing > 0)

    df['estado']     = df['cve_ent'].map(ENTIDADES)  if 'cve_ent'   in df.columns else None
    df['sexo']       = df['sex'].map(SEXO)            if 'sex'       in df.columns else None
    df['sector']     = df['rama_est2'].map(SECTORES)  if 'rama_est2' in df.columns else None
    df['nivel_educ'] = df['niv_ins'].map(NIV_INS)     if 'niv_ins'   in df.columns else None
    df['ocupacion']  = df['c_ocu11c'].map(OCUPACION)  if 'c_ocu11c'  in df.columns else None
    df['formalidad'] = df['emp_ppal'].map(FORMAL)      if 'emp_ppal'  in df.columns else None

    return df

## Proceso principal

Por cada archivo crudo: si el filtrado ya existe en la carpeta de salida se omite.

In [7]:
archivos = []
for carpeta in CARPETAS_ENTRADA:
    if not os.path.exists(carpeta):
        print(f'  Carpeta no encontrada, se omite: {carpeta}')
        continue
    encontrados = sorted(glob.glob(os.path.join(carpeta, '*SDEMT*.csv')))
    print(f'  {len(encontrados):>3} archivos en {carpeta}')
    archivos.extend(encontrados)

print(f'\nTotal: {len(archivos)} archivos')

   80 archivos en C:\Users\andre\hackODS\LoboEnsambladores\datos\crudo

Total: 80 archivos


In [8]:
for ruta in archivos:
    nf = nombre_filtrado(ruta)

    if nf is None:
        print(f'  OMITIDO (nombre no reconocido): {os.path.basename(ruta)}')
        continue

    salida = os.path.join(CARPETA_SALIDA, nf)

    if os.path.exists(salida):
        print(f'  ya existe: {nf}')
        continue

    try:
        df = filtrar(ruta)
        df.to_csv(salida, index=False, encoding='utf-8-sig')
        print(f'  {os.path.basename(ruta)} -> {nf}  ({len(df):,} filas, {df.shape[1]} cols)')
        del df
        gc.collect()
    except Exception as e:
        print(f'  ERROR en {os.path.basename(ruta)}: {e}')

print('\nProceso terminado.')

  ENOEN_SDEMT121.csv -> ENOE_T121.csv  (148,991 filas, 20 cols)
  ENOEN_SDEMT122.csv -> ENOE_T122.csv  (177,295 filas, 20 cols)
  ENOEN_SDEMT221.csv -> ENOE_T221.csv  (167,677 filas, 20 cols)
  ENOEN_SDEMT222.csv -> ENOE_T222.csv  (182,163 filas, 20 cols)
  ENOEN_SDEMT320.csv -> ENOE_T320.csv  (120,316 filas, 20 cols)
  ENOEN_SDEMT321.csv -> ENOE_T321.csv  (188,379 filas, 20 cols)
  ENOEN_SDEMT322.csv -> ENOE_T322.csv  (179,396 filas, 20 cols)
  ENOEN_SDEMT420.csv -> ENOE_T420.csv  (152,033 filas, 20 cols)
  ENOEN_SDEMT422.csv -> ENOE_T422.csv  (180,687 filas, 20 cols)
  ya existe: ENOE_T120.csv
  ENOE_SDEMT123.csv -> ENOE_T123.csv  (204,787 filas, 20 cols)
  ENOE_SDEMT124.csv -> ENOE_T124.csv  (194,261 filas, 20 cols)
  ENOE_SDEMT125.csv -> ENOE_T125.csv  (190,503 filas, 20 cols)
  ENOE_SDEMT224.csv -> ENOE_T224.csv  (196,600 filas, 20 cols)
  ENOE_SDEMT225.csv -> ENOE_T225.csv  (191,843 filas, 20 cols)
  ENOE_SDEMT323.csv -> ENOE_T323.csv  (197,262 filas, 20 cols)
  ENOE_SDEMT324.csv

In [9]:
filtrados = sorted(os.listdir(CARPETA_SALIDA))
total_mb = 0
print(f'Archivos en filtrado/ ({len(filtrados)} total):\n')
for f in filtrados:
    mb = os.path.getsize(os.path.join(CARPETA_SALIDA, f)) / 1_048_576
    total_mb += mb
    print(f'  {f:<26}  {mb:5.1f} MB')
print(f'\n  Total: {total_mb:.1f} MB')

Archivos en filtrado/ (80 total):

  ENOE_T105.csv                20.3 MB
  ENOE_T106.csv                21.2 MB
  ENOE_T107.csv                21.3 MB
  ENOE_T108.csv                21.2 MB
  ENOE_T109.csv                20.3 MB
  ENOE_T110.csv                20.3 MB
  ENOE_T111.csv                20.0 MB
  ENOE_T112.csv                20.4 MB
  ENOE_T113.csv                19.9 MB
  ENOE_T114.csv                20.4 MB
  ENOE_T115.csv                20.6 MB
  ENOE_T116.csv                20.4 MB
  ENOE_T117.csv                20.3 MB
  ENOE_T118.csv                20.4 MB
  ENOE_T119.csv                21.5 MB
  ENOE_T120.csv                22.4 MB
  ENOE_T121.csv                18.1 MB
  ENOE_T122.csv                21.5 MB
  ENOE_T123.csv                24.8 MB
  ENOE_T124.csv                23.5 MB
  ENOE_T125.csv                22.8 MB
  ENOE_T205.csv                20.6 MB
  ENOE_T206.csv                21.3 MB
  ENOE_T207.csv                21.4 MB
  ENOE_T208.csv              